# OWASP Dependency-Check — Java SCA raw output and metric mapping

This notebook:
1. **Discovers** Java build artifacts and dependency inputs under a configurable project path (`pom.xml`, `build.gradle`, `*.jar`, etc.).
2. **Runs** the [OWASP Dependency-Check](https://dependency-check.github.io/DependencyCheck/) CLI against that path.
3. **Shows** CLI stdout/stderr and the **raw JSON report** (pretty-printed).
4. **Maps** report fields to your dependency-risk metrics (right column) and states what Dependency-Check does **not** provide.

## Your metrics vs Dependency-Check

| Metric | In raw Dependency-Check output? |
|--------|--------------------------------|
| **Known CVE Count** | Yes — `dependencies[].vulnerabilities` |
| **Hidden Relationship Mapping** | Partially — `relatedDependencies`, dependency graph context; full transitive tree is clearer via Maven/Gradle + DC together |
| **Legal Risk Validation** | Partially — license fields when present (`packages` / evidence); not as complete as dedicated license scanners |
| **Trust Integrity Verification** | Partially — hashes (`md5`, `sha1`, `sha256`); not full sigstore/provenance |
| **Community Vitality Tracking** | No — needs external APIs (e.g. GitHub stars, release dates) |
| **Mitigation Effort Ranking** | Partially — CVSS/severity supports *priority*; “effort to fix” is not a first-class field |
| **Real-Time Alerting** | No — batch CLI; use CI schedules, webhooks, or a commercial/SaaS layer |
| **Version Lag Assessment** | Partially — current GAV in evidence/packages; “latest version” may appear in some report versions / analyzers — verify in your JSON |

## Prerequisites

- **JDK** installed (`java -version`).
- **OWASP Dependency-Check CLI**: download from [releases](https://github.com/dependency-check/DependencyCheck/releases), unzip, and set `DEPENDENCY_CHECK_HOME` below (folder containing `dependency-check.bat` on Windows).
- **NVD API key** (recommended): [request from NIST](https://nvd.nist.gov/developers/request-an-api-key); set `NVD_API_KEY` environment variable or pass `--nvdApiKey` in the command.

Install Python packages for this notebook:

```bash
pip install -r requirements-notebook.txt
```

## 1. Configuration

In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

_base = Path.cwd().resolve()

# Root of the Java project to scan (override with JAVA_PROJECT_ROOT)
PROJECT_ROOT = Path(os.environ.get("JAVA_PROJECT_ROOT", _base / "sample-java-app")).resolve()

# Unzipped OWASP Dependency-Check CLI (override with DEPENDENCY_CHECK_HOME)
DEPENDENCY_CHECK_HOME = Path(
    os.environ.get("DEPENDENCY_CHECK_HOME", _base / "tools" / "dependency-check")
).resolve()

# Report output directory (created by this notebook run)
REPORT_DIR = Path.cwd() / "dc_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

JSON_REPORT = REPORT_DIR / "dependency-check-report.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEPENDENCY_CHECK_HOME:", DEPENDENCY_CHECK_HOME)
print("REPORT_DIR:", REPORT_DIR)

PROJECT_ROOT: F:\java_metrics\sample-java-app
DEPENDENCY_CHECK_HOME: F:\java_metrics\tools\dependency-check
REPORT_DIR: F:\java_metrics\dc_reports


## 2. List Java-related files under the project (inputs to SCA)

Dependency-Check consumes build descriptors, JARs, and archives—not only `.java` sources. This cell lists what will be *visible* on disk for scanning.

In [2]:
JAVA_SOURCE_PATTERNS = ("*.java",)
BUILD_PATTERNS = ("pom.xml", "build.gradle", "build.gradle.kts", "settings.gradle", "settings.gradle.kts")
ARCHIVE_PATTERNS = ("*.jar", "*.war", "*.ear")

if not PROJECT_ROOT.is_dir():
    print(f"WARNING: PROJECT_ROOT does not exist: {PROJECT_ROOT}")
    print("Create or point JAVA_PROJECT_ROOT to a real Maven/Gradle project before running the scan.")
else:
    java_files = []
    for pat in JAVA_SOURCE_PATTERNS:
        java_files.extend(PROJECT_ROOT.rglob(pat))
    build_files = []
    for name in BUILD_PATTERNS:
        build_files.extend(PROJECT_ROOT.rglob(name))
    archives = []
    for pat in ARCHIVE_PATTERNS:
        archives.extend(PROJECT_ROOT.rglob(pat))

    # Skip heavy dirs
    def skip(p: Path) -> bool:
        parts = set(p.parts)
        return "target" in parts or "build" in parts or ".git" in parts

    java_files = [p for p in java_files if not skip(p)]
    build_files = [p for p in build_files if not skip(p)]
    archives = [p for p in archives if not skip(p)]

    print(f"Java sources found: {len(java_files)} (showing up to 30)")
    for p in sorted(java_files)[:30]:
        print("  ", p.relative_to(PROJECT_ROOT))
    if len(java_files) > 30:
        print("  ...")
    print(f"\nBuild descriptors: {len(build_files)}")
    for p in sorted(build_files):
        print("  ", p.relative_to(PROJECT_ROOT))
    print(f"\nArchives (jar/war/ear, excluding target/build): {len(archives)}")
    for p in sorted(archives)[:20]:
        print("  ", p.relative_to(PROJECT_ROOT))
    if len(archives) > 20:
        print("  ...")

Java sources found: 1 (showing up to 30)
   src\main\java\com\example\App.java

Build descriptors: 1
   pom.xml

Archives (jar/war/ear, excluding target/build): 1
   lib\commons-collections-3.2.1.jar


## 3. Resolve Dependency-Check executable

In [3]:
if sys.platform == "win32":
    DC_EXE = DEPENDENCY_CHECK_HOME / "bin" / "dependency-check.bat"
else:
    DC_EXE = DEPENDENCY_CHECK_HOME / "bin" / "dependency-check.sh"

if not DC_EXE.is_file():
    raise FileNotFoundError(
        f"Dependency-Check not found at {DC_EXE}. "
        "Set DEPENDENCY_CHECK_HOME to the unzipped CLI directory."
    )
print("Using:", DC_EXE)

Using: F:\java_metrics\tools\dependency-check\bin\dependency-check.bat


## 4. Run OWASP Dependency-Check (raw CLI output)

Adjust flags as needed: `--enableExperimental`, `--nodePackageSkipDevDependencies`, etc. See `dependency-check.sh --help`.

**First run** downloads the NVD CVE database and can take **15–45+ minutes** without an NVD API key. After `data/odc.mv.db` exists under `DEPENDENCY_CHECK_HOME`, the notebook adds `--noupdate` automatically so later runs finish in about a minute unless you set `DEPENDENCY_CHECK_FORCE_UPDATE=1`.

In [ ]:
nvd_key = os.environ.get("NVD_API_KEY", "")
nvd_args = ["--nvdApiKey", nvd_key] if nvd_key else []

dc_data_db = DEPENDENCY_CHECK_HOME / "data" / "odc.mv.db"
force_update = os.environ.get("DEPENDENCY_CHECK_FORCE_UPDATE", "").lower() in ("1", "true", "yes")
noupdate = (not force_update) and (
    dc_data_db.is_file()
    or os.environ.get("DEPENDENCY_CHECK_NOUPDATE", "").lower() in ("1", "true", "yes")
)
update_args = ["-n"] if noupdate else []
if noupdate:
    print("Using --noupdate (local CVE DB present or DEPENDENCY_CHECK_NOUPDATE set).")
elif not dc_data_db.is_file():
    print("No local CVE DB yet; this run may take a long time while NVD data is downloaded.")

cmd = [
    str(DC_EXE),
] + update_args + [
    "--scan",
    str(PROJECT_ROOT),
    "--format",
    "JSON",
    "--format",
    "HTML",
    "--out",
    str(REPORT_DIR),
    "--project",
    PROJECT_ROOT.name,
] + nvd_args

safe_cmd = [("***" if str(c) == nvd_key else str(c)) for c in cmd] if nvd_key else [str(c) for c in cmd]
print("Command:", " ".join(safe_cmd))

proc = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    shell=False,
)

print("\n=== STDOUT ===\n")
print(proc.stdout or "(empty)")
print("\n=== STDERR ===\n")
print(proc.stderr or "(empty)")
print("\nExit code:", proc.returncode)

if proc.returncode != 0:
    print(
        "\nNote: Dependency-Check often exits non-zero when vulnerabilities are found. "
        "Check whether JSON was still written."
    )

Using --noupdate (local CVE DB present or DEPENDENCY_CHECK_NOUPDATE set).
Command: F:\java_metrics\tools\dependency-check\bin\dependency-check.bat -n --scan F:\java_metrics\sample-java-app --format JSON --format HTML --out F:\java_metrics\dc_reports --project sample-java-app



=== STDOUT ===

[INFO] 

Dependency-Check is an open source tool performing a best effort analysis of 3rd party dependencies; false positives and false negatives may exist in the analysis performed by the tool. Use of the tool and the reporting provided constitutes acceptance for use in an AS IS condition, and there are NO warranties, implied or otherwise, with regard to the analysis or its use. Any use of the tool and the reporting provided is at the user's risk. In no event shall the copyright holder or OWASP be held liable for any damages whatsoever arising out of or in connection with the use of this tool, the analysis performed, or the resulting report.


   About ODC: https://dependency-check.github.io/DependencyCheck/general/internals.html
   False Positives: https://dependency-check.github.io/DependencyCheck/general/suppression.html


[INFO] Analysis Started
[INFO] Finished Archive Analyzer (0 seconds)
[INFO] Finished File Name Analyzer (0 seconds)
[INFO] Finished Jar Analyzer

## 5. Raw JSON report (full document)

If the file is large, use the summarization cell next instead of printing everything.

In [5]:
if not JSON_REPORT.is_file():
    raise FileNotFoundError(
        f"Expected report not found: {JSON_REPORT}. "
        "Fix CLI errors above or run the scan from a valid project path."
    )

raw_text = JSON_REPORT.read_text(encoding="utf-8", errors="replace")
report = json.loads(raw_text)

MAX_CHARS = 120_000
pretty = json.dumps(report, indent=2)
if len(pretty) > MAX_CHARS:
    print(pretty[:MAX_CHARS])
    print(f"\n... truncated ({len(pretty)} chars total). Full file: {JSON_REPORT}")
else:
    print(pretty)

{
  "reportSchema": "1.1",
  "scanInfo": {
    "engineVersion": "12.2.0",
    "dataSource": []
  },
  "projectInfo": {
    "name": "sample-java-app",
    "reportDate": "2026-03-27T17:12:36.717106200Z",
    "credits": {
      "NVD": "This product uses the NVD API but is not endorsed or certified by the NVD. This report contains data retrieved from the National Vulnerability Database: https://nvd.nist.gov",
      "CISA": "This report may contain data retrieved from the CISA Known Exploited Vulnerability Catalog: https://www.cisa.gov/known-exploited-vulnerabilities-catalog",
      "NPM": "This report may contain data retrieved from the Github Advisory Database (via NPM Audit API): https://github.com/advisories/",
      "RETIREJS": "This report may contain data retrieved from the RetireJS community: https://retirejs.github.io/retire.js/",
      "OSSINDEX": "This report may contain data retrieved from the Sonatype OSS Index: https://ossindex.sonatype.org"
    }
  },
  "dependencies": [
    

## 6. Metric-oriented slices (derived from raw JSON)

These extract fields that align with your **right-column** metrics. Field names vary slightly across Dependency-Check versions; this cell uses defensive `.get()` access.

In [6]:
deps = report.get("dependencies") or []

# --- Known CVE Count ---
total_cves = 0
cve_by_dep = []
for d in deps:
    vulns = d.get("vulnerabilities") or []
    total_cves += len(vulns)
    names = [v.get("name") for v in vulns if v.get("name")]
    if names:
        cve_by_dep.append({"fileName": d.get("fileName"), "cves": names})

print("=== Known CVE Count ===")
print("Total vulnerability records:", total_cves)
print("Dependencies with at least one CVE:", len(cve_by_dep))
for row in cve_by_dep[:15]:
    print(row["fileName"], "->", ", ".join(row["cves"][:5]), "..." if len(row["cves"]) > 5 else "")
if len(cve_by_dep) > 15:
    print("...")

# --- Hidden Relationship Mapping (transitive / related) ---
print("\n=== Hidden Relationship Mapping (relatedDependencies) ===")
related_count = 0
for d in deps:
    rel = d.get("relatedDependencies") or []
    if rel:
        related_count += len(rel)
        print(d.get("fileName"), "related:", len(rel))
        for r in rel[:3]:
            print("   ", r)
if related_count == 0:
    print("No relatedDependencies blocks in this report (common for some analyzers). "
          "Use Maven `dependency:tree` or Gradle `dependencies` for full transitive view.")

# --- Legal Risk Validation (licenses) ---
print("\n=== Legal Risk Validation (license hints in report) ===")
for d in deps[:50]:
    packages = d.get("packages") or []
    for pkg in packages:
        lic = pkg.get("license")
        if lic:
            print(d.get("fileName"), pkg.get("id"), "license:", lic)
evidence = None
for d in deps:
    ev = d.get("evidenceCollected") or {}
    if "licenses" in ev:
        evidence = ev["licenses"]
        print("Evidence licenses sample:", evidence[:5] if isinstance(evidence, list) else evidence)
        break

# --- Trust Integrity Verification (hashes) ---
print("\n=== Trust Integrity Verification (hashes) ===")
for d in deps[:10]:
    print(
        d.get("fileName"),
        "sha256:", (d.get("sha256") or "")[:16] + "..." if d.get("sha256") else None,
    )

# --- Mitigation Effort Ranking (severity proxy) ---
print("\n=== Mitigation Effort Ranking (CVSS / severity as priority proxy) ===")
ranked = []
for d in deps:
    for v in d.get("vulnerabilities") or []:
        ranked.append(
            {
                "cve": v.get("name"),
                "severity": v.get("severity"),
                "cvssv3": (v.get("cvssv3") or {}).get("baseScore"),
                "dependency": d.get("fileName"),
            }
        )
ranked.sort(key=lambda x: (x["cvssv3"] is None, -(x["cvssv3"] or 0)))
for r in ranked[:20]:
    print(r)

# --- Version Lag Assessment (best-effort: package IDs contain version) ---
print("\n=== Version Lag Assessment ===")
print("Dependency-Check JSON usually encodes current GAV in package coordinates (pkg:maven/...).")
print("Comparing to *latest* requires Maven Central or similar — not always in this JSON.")
for d in deps[:15]:
    pkgs = d.get("packages") or []
    if pkgs:
        print(d.get("fileName"), "->", pkgs[0].get("id"))

# --- Community Vitality & Real-Time Alerting ---
print("\n=== Community Vitality Tracking / Real-Time Alerting ===")
print("Not produced by Dependency-Check CLI. Add: GitHub API, OSSF Scorecard, or CI + notifications.")

=== Known CVE Count ===


Total vulnerability records: 1
Dependencies with at least one CVE: 1
commons-collections-3.2.1.jar -> CVE-2015-6420 

=== Hidden Relationship Mapping (relatedDependencies) ===
No relatedDependencies blocks in this report (common for some analyzers). Use Maven `dependency:tree` or Gradle `dependencies` for full transitive view.

=== Legal Risk Validation (license hints in report) ===

=== Trust Integrity Verification (hashes) ===
commons-collections-3.2.1.jar sha256: 87363a4c94eaabee...

=== Mitigation Effort Ranking (CVSS / severity as priority proxy) ===
{'cve': 'CVE-2015-6420', 'severity': 'CRITICAL', 'cvssv3': 9.8, 'dependency': 'commons-collections-3.2.1.jar'}

=== Version Lag Assessment ===
Dependency-Check JSON usually encodes current GAV in package coordinates (pkg:maven/...).
Comparing to *latest* requires Maven Central or similar — not always in this JSON.
commons-collections-3.2.1.jar -> pkg:maven/commons-collections/commons-collections@3.2.1

=== Community Vitality Tracking

## 7. Open HTML report (optional)

After a successful run, open `dependency-check-report.html` under `REPORT_DIR` in a browser for the full UI.

In [7]:
html_path = REPORT_DIR / "dependency-check-report.html"
if html_path.is_file():
    print(html_path.resolve().as_uri())
else:
    print("HTML report not found:", html_path)

file:///F:/java_metrics/dc_reports/dependency-check-report.html
